In [1]:
import re
import json
import csv
from pathlib import Path

# =========================
# 설정
# =========================
BASE       = Path(r'C:\Users\82105\OneDrive\바탕 화면\프로젝트3(면접)')
INPUT_DIR  = BASE / '샘플영상' / '이형 유튜브 영상' / 'DB용'   # TXT 파일 위치
OUTPUT_DIR = INPUT_DIR / 'gemini_output'                         # 출력 폴더

TRAIN_JSONL = 'gemini_train.jsonl'   # Gemini 튜닝 형식 (text_input / output)
PREVIEW_CSV = 'preview.csv'

MIN_ANSWER_LEN   = 5
MIN_FEEDBACK_LEN = 5

# Gemini 튜닝용 시스템 프롬프트 (text_input 앞에 포함)
SYSTEM_PROMPT = """당신은 대기업 인사담당자 출신의 직설적인 면접 코치 '이형'이다.
10년간 수천 명을 면접했고, 지원자 답변을 들으면 합격/불합격이 바로 보인다.
친근한 말투를 쓰되 평가는 냉정하게 한다.
반드시 아래 형식으로만 답한다:

[이형의 팩폭 한줄평]
한 문장으로 이 답변의 핵심 문제 또는 강점을 직격한다.

[이형의 시선]
면접관 관점에서 이 답변이 어떻게 들리는지, 왜 좋은지/나쁜지 구체적으로 분석한다. (3~5문장)

[이형의 합격 처방전]
1. 즉시 실천 가능한 구체적 개선 방법
2. 답변 구조/내용 개선 방법
3. 면접관에게 어필할 포인트"""

# =========================
# 유틸
# =========================
def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

def read_txt_auto(path: Path) -> str:
    for enc in ['utf-8-sig', 'utf-8', 'cp949', 'euc-kr']:
        try:
            return path.read_text(encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f'인코딩 읽기 실패: {path}')

def clean_text(text: str) -> str:
    text = text.replace('\xa0', ' ')
    text = re.sub(r'\r\n?', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def normalize_spaces(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip()

# =========================
# 블록 분리
# =========================
def split_answer_blocks(text):
    pattern = r'(답변\s*\d+\s*[\.\)]\s*.*?)(?=(?:\n답변\s*\d+\s*[\.\)])|\Z)'
    return [b.strip() for b in re.findall(pattern, text, flags=re.DOTALL) if b.strip()]

def split_af_blocks(text):
    pattern = r'(A\s*[.:：]\s*.*?F\s*[.:：]\s*.*?)(?=(?:\nA\s*[.:：]\s*)|\Z)'
    return [b.strip() for b in re.findall(pattern, text, flags=re.DOTALL) if b.strip()]

def parse_answer_block(block):
    first_line = block.split('\n', 1)[0].strip()
    question = re.sub(r'^답변\s*\d+\s*[\.\)]\s*', '', first_line).strip()
    for p in [r'\nA\.\s*(.*?)(?:\nF\.\s*)(.*)$', r'\nA\s*[:：]\s*(.*?)(?:\nF\s*[:：]\s*)(.*)$']:
        m = re.search(p, block, flags=re.DOTALL)
        if m:
            return {'question': question or '없음', 'answer': m.group(1).strip(), 'feedback': m.group(2).strip()}
    return None

def parse_af_block(block, idx):
    for p in [r'A\.\s*(.*?)(?:\nF\.\s*)(.*)$', r'A\s*[:：]\s*(.*?)(?:\nF\s*[:：]\s*)(.*)$']:
        m = re.search(p, block, flags=re.DOTALL)
        if m:
            return {'question': f'질문 미상_{idx}', 'answer': m.group(1).strip(), 'feedback': m.group(2).strip()}
    return None

# =========================
# 유효성 검사
# =========================
def has_valid_answer(answer):
    return bool(answer) and len(normalize_spaces(answer)) >= MIN_ANSWER_LEN

def has_valid_feedback(feedback):
    if not feedback:
        return False
    compact = normalize_spaces(feedback)
    invalid = {'직접적으로 언급된 피드백은 없음', '직접 언급된 피드백은 없음', '피드백은 없음', '언급된 피드백은 없음', '없음'}
    if compact in invalid:
        return False
    if '피드백은 없음' in compact and len(compact) < 40:
        return False
    cleaned = re.sub(r'[\"""\'`.,!?…\-]', '', compact).strip()
    return len(cleaned) >= MIN_FEEDBACK_LEN

# =========================
# 피드백 형식화 (이형 3섹션)
# =========================
def split_sentences(text):
    text = normalize_spaces(text.replace('\n', ' '))
    for ep in ['입니다.', '합니다.', '했어요.', '좋아요.', '같아요.', '거든요.', '있어요.', '없어요.', '하세요.']:
        text = text.replace(ep + ' ', ep + '|||')
    text = re.sub(r'([.!?])\s+', r'\1|||', text)
    return [p.strip(' \"""\'') for p in text.split('|||') if p.strip()]

def build_assistant_feedback(raw_feedback):
    compact = raw_feedback.strip()
    # 이미 이형 형식이면 그대로 사용
    if '[이형의 팩폭 한줄평]' in compact and '[이형의 시선]' in compact and '[이형의 합격 처방전]' in compact:
        return compact
    sentences = split_sentences(compact)
    if not sentences:
        return None
    one_liner = sentences[0]
    if len(sentences) == 1:
        analysis = sentences[0]
        prescription = '1. 답변의 핵심 근거를 더 구체화하세요.\n2. 면접관이 가질 의문을 먼저 해소하세요.\n3. 마지막은 회사와 직무에 대한 연결로 마무리하세요.'
    elif len(sentences) == 2:
        analysis = ' '.join(sentences)
        prescription = '1. 장점은 유지하되 부족한 연결 고리를 보완하세요.\n2. 추상적인 표현 대신 근거를 더 분명히 하세요.\n3. 왜 이 회사인지가 보이도록 마무리하세요.'
    else:
        analysis = ' '.join(sentences[:-1])
        prescription = f'1. {sentences[-1]}\n2. 답변의 근거를 더 구체적으로 제시하세요.\n3. 마지막은 회사와 직무에 대한 연결로 닫으세요.'
    return f'[이형의 팩폭 한줄평]\n{one_liner}\n\n[이형의 시선]\n{analysis}\n\n[이형의 합격 처방전]\n{prescription}'

# =========================
# Gemini 튜닝 형식 row
# =========================
def make_row(question, answer, assistant_feedback):
    text_input = f'{SYSTEM_PROMPT}\n\n면접 질문: {question}\n지원자 답변: {answer}'
    return {'text_input': text_input, 'output': assistant_feedback}

# =========================
# 저장
# =========================
def save_jsonl(path, rows):
    with open(path, 'w', encoding='utf-8') as f:   # BOM 없는 UTF-8
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

def save_csv(path, rows):
    fieldnames = ['source_file', 'sample_id', 'question', 'answer_preview', 'raw_feedback_preview', 'output_preview']
    with open(path, 'w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

# =========================
# 메인
# =========================
def main():
    ensure_dir(OUTPUT_DIR)
    txt_files = sorted(INPUT_DIR.glob('*.txt'))   # DB용 폴더의 TXT만 (하위폴더 제외)
    if not txt_files:
        print(f'TXT 파일이 없습니다: {INPUT_DIR}')
        return

    all_records, preview_rows = [], []
    total_blocks = parsed_blocks = skipped_no_parse = skipped_invalid = 0

    for file_path in txt_files:
        try:
            text = clean_text(read_txt_auto(file_path))
            blocks = split_answer_blocks(text)
            mode = 'answer'
            if not blocks:
                blocks = split_af_blocks(text)
                mode = 'af'
            print(f'[처리중] {file_path.name} | 모드:{mode} | 블록:{len(blocks)}')

            saved = skipped = 0
            for idx, block in enumerate(blocks, 1):
                total_blocks += 1
                parsed = parse_answer_block(block) if mode == 'answer' else parse_af_block(block, idx)
                if not parsed:
                    skipped_no_parse += 1; skipped += 1; continue
                parsed_blocks += 1

                q, a, raw_fb = parsed['question'], parsed['answer'], parsed['feedback']
                if not has_valid_answer(a) or not has_valid_feedback(raw_fb):
                    skipped_invalid += 1; skipped += 1; continue

                assistant_fb = build_assistant_feedback(raw_fb)
                if not assistant_fb:
                    skipped_invalid += 1; skipped += 1; continue

                row = make_row(q, a, assistant_fb)
                all_records.append(row)
                preview_rows.append({
                    'source_file': file_path.name,
                    'sample_id': f'{file_path.stem}_{idx}',
                    'question': q,
                    'answer_preview': a[:200],
                    'raw_feedback_preview': raw_fb[:200],
                    'output_preview': assistant_fb[:200]
                })
                saved += 1
            print(f'  → 저장:{saved}개  제외:{skipped}개')
        except Exception as e:
            print(f'[에러] {file_path.name}: {e}')

    if not all_records:
        print('유효한 샘플이 없습니다.')
        return

    save_jsonl(OUTPUT_DIR / TRAIN_JSONL, all_records)
    save_csv(OUTPUT_DIR / PREVIEW_CSV, preview_rows)

    print(f'\n=== 완료 ===')
    print(f'전체블록:{total_blocks}  저장:{len(all_records)}  파싱실패:{skipped_no_parse}  유효성실패:{skipped_invalid}')
    print(f'출력: {OUTPUT_DIR / TRAIN_JSONL}')

main()

[처리중] 1.txt | 모드:af | 블록:1
  → 저장:1개  제외:0개
[처리중] 10.txt | 모드:answer | 블록:5
  → 저장:5개  제외:0개
[처리중] 13.txt | 모드:answer | 블록:2
  → 저장:2개  제외:0개
[처리중] 14.txt | 모드:answer | 블록:3
  → 저장:3개  제외:0개
[처리중] 15.txt | 모드:answer | 블록:2
  → 저장:2개  제외:0개
[처리중] 16.txt | 모드:answer | 블록:4
  → 저장:4개  제외:0개
[처리중] 17.txt | 모드:answer | 블록:14
  → 저장:11개  제외:3개
[처리중] 18.txt | 모드:answer | 블록:14
  → 저장:11개  제외:3개
[처리중] 19.txt | 모드:answer | 블록:8
  → 저장:8개  제외:0개
[처리중] 2.txt | 모드:af | 블록:1
  → 저장:1개  제외:0개
[처리중] 20.txt | 모드:answer | 블록:6
  → 저장:6개  제외:0개
[처리중] 3.txt | 모드:af | 블록:1
  → 저장:1개  제외:0개
[처리중] 4.txt | 모드:af | 블록:1
  → 저장:1개  제외:0개
[처리중] 5.txt | 모드:af | 블록:2
  → 저장:2개  제외:0개
[처리중] 6.txt | 모드:answer | 블록:3
  → 저장:3개  제외:0개
[처리중] 7.txt | 모드:answer | 블록:3
  → 저장:3개  제외:0개
[처리중] 8.txt | 모드:answer | 블록:3
  → 저장:3개  제외:0개
[처리중] 9.txt | 모드:answer | 블록:1
  → 저장:1개  제외:0개

=== 완료 ===
전체블록:74  저장:68  파싱실패:0  유효성실패:6
출력: C:\Users\82105\OneDrive\바탕 화면\프로젝트3(면접)\샘플영상\이형 유튜브 영상\DB용\gemini_output\gemini_train.jsonl
